Link to the notebook: https://colab.research.google.com/drive/1PHiXvgxOsn_G5vh8RnXHSFrhkU_2-Lb3?usp=sharing

In [1]:
!curl -LsSf https://astral.sh/uv/install.sh | sh

downloading uv 0.11.3 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!


In [2]:
!uv pip install requests

Using Python 3.12.13 environment at: /usr
Checked 1 package in 108ms


# Download SMD

In [3]:
# Imports

from pathlib import Path
import requests

In [4]:
# Repository configuration

DATASET_REPOSITORY_OWNER = "NetManAIOps"
DATASET_REPOSITORY_NAME = "OmniAnomaly"
DATASET_REPOSITORY_BRANCH = "master"
DATASET_DIRECTORY_IN_REPOSITORY = "ServerMachineDataset"

LOCAL_OUTPUT_DIRECTORY = Path("ServerMachineDataset")

In [5]:
# Expected dataset structure for a small validation check

EXPECTED_TOP_LEVEL_CHILDREN = {
    "LICENSE",
    "train",
    "test",
    "test_label",
    "interpretation_label",
}

In [6]:
# Create an HTTP session explicitly

http_session = requests.Session()
http_session.headers.update(
    {
        "Accept": "application/vnd.github+json",
        "User-Agent": "smd-dataset-downloader",
    }
)

http_session.headers

{'User-Agent': 'smd-dataset-downloader', 'Accept-Encoding': 'gzip, deflate, br, zstd', 'Accept': 'application/vnd.github+json', 'Connection': 'keep-alive'}

In [7]:
# Build the GitHub Contents API URL for the dataset directory

github_api_url = (
    f"https://api.github.com/repos/"
    f"{DATASET_REPOSITORY_OWNER}/"
    f"{DATASET_REPOSITORY_NAME}/contents/"
    f"{DATASET_DIRECTORY_IN_REPOSITORY}"
    f"?ref={DATASET_REPOSITORY_BRANCH}"
)

github_api_url

'https://api.github.com/repos/NetManAIOps/OmniAnomaly/contents/ServerMachineDataset?ref=master'

In [8]:
# Request metadata for the top-level dataset directory

top_level_response = http_session.get(github_api_url, timeout=30)
top_level_response.raise_for_status()

top_level_items = top_level_response.json()
type(top_level_items), len(top_level_items)

(list, 5)

In [9]:
# Inspect the top-level directory items returned by GitHub

for item in top_level_items:
    print(
        "name =", item["name"],
        "| type =", item["type"],
        "| path =", item["path"]
    )

name = LICENSE | type = file | path = ServerMachineDataset/LICENSE
name = interpretation_label | type = dir | path = ServerMachineDataset/interpretation_label
name = test | type = dir | path = ServerMachineDataset/test
name = test_label | type = dir | path = ServerMachineDataset/test_label
name = train | type = dir | path = ServerMachineDataset/train


In [10]:
# Create the local output directory

LOCAL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
LOCAL_OUTPUT_DIRECTORY.resolve()

PosixPath('/content/ServerMachineDataset')

In [11]:
# Helper: download one file from a raw GitHub URL

def download_binary_file(file_download_url: str, local_output_file_path: Path) -> None:
    local_output_file_path.parent.mkdir(parents=True, exist_ok=True)

    file_response = http_session.get(file_download_url, timeout=60)
    file_response.raise_for_status()

    local_output_file_path.write_bytes(file_response.content)
    print(f"Saved file: {local_output_file_path}")

In [12]:
# Helper: build a GitHub Contents API URL for any repository path

def build_github_contents_api_url(path_in_repository: str) -> str:
    return (
        f"https://api.github.com/repos/"
        f"{DATASET_REPOSITORY_OWNER}/"
        f"{DATASET_REPOSITORY_NAME}/contents/"
        f"{path_in_repository}"
        f"?ref={DATASET_REPOSITORY_BRANCH}"
    )

In [13]:
# Helper: request JSON metadata for any repository path

def request_github_json(path_in_repository: str):
    api_url = build_github_contents_api_url(path_in_repository)
    response = http_session.get(api_url, timeout=30)
    response.raise_for_status()
    return response.json()

In [14]:
# Inspect one subdirectory manually before recursion
# You can change "train" to "test", "test_label", or "interpretation_label"

example_subdirectory_path = "ServerMachineDataset/train"
example_subdirectory_items = request_github_json(example_subdirectory_path)

for item in example_subdirectory_items[:5]:
    print(
        "name =", item["name"],
        "| type =", item["type"],
        "| download_url =", item.get("download_url")
    )

name = machine-1-1.txt | type = file | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/train/machine-1-1.txt
name = machine-1-2.txt | type = file | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/train/machine-1-2.txt
name = machine-1-3.txt | type = file | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/train/machine-1-3.txt
name = machine-1-4.txt | type = file | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/train/machine-1-4.txt
name = machine-1-5.txt | type = file | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/train/machine-1-5.txt


In [15]:
# Re-check the top-level GitHub payload shape very explicitly

print(type(top_level_items))
print(len(top_level_items) if isinstance(top_level_items, list) else "not a list")
print(top_level_items[:2] if isinstance(top_level_items, list) else top_level_items)

<class 'list'>
5
[{'name': 'LICENSE', 'path': 'ServerMachineDataset/LICENSE', 'sha': 'a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'size': 1072, 'url': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/contents/ServerMachineDataset/LICENSE?ref=master', 'html_url': 'https://github.com/NetManAIOps/OmniAnomaly/blob/master/ServerMachineDataset/LICENSE', 'git_url': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/git/blobs/a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'download_url': 'https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/LICENSE', 'type': 'file', '_links': {'self': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/contents/ServerMachineDataset/LICENSE?ref=master', 'git': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/git/blobs/a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'html': 'https://github.com/NetManAIOps/OmniAnomaly/blob/master/ServerMachineDataset/LICENSE'}}, {'name': 'interpretation_label', 'path': 'ServerMachineDataset/i

In [16]:
# Print only the fields we actually rely on

for index, item in enumerate(top_level_items):
    print(
        index,
        "| name =", item.get("name"),
        "| type =", item.get("type"),
        "| path =", item.get("path"),
        "| download_url =", item.get("download_url"),
    )

0 | name = LICENSE | type = file | path = ServerMachineDataset/LICENSE | download_url = https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/LICENSE
1 | name = interpretation_label | type = dir | path = ServerMachineDataset/interpretation_label | download_url = None
2 | name = test | type = dir | path = ServerMachineDataset/test | download_url = None
3 | name = test_label | type = dir | path = ServerMachineDataset/test_label | download_url = None
4 | name = train | type = dir | path = ServerMachineDataset/train | download_url = None


In [17]:
# Reset the local directory so the test is clean

import shutil

if LOCAL_OUTPUT_DIRECTORY.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIRECTORY)

LOCAL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Reset directory:", LOCAL_OUTPUT_DIRECTORY.resolve())
print("Children after reset:", list(LOCAL_OUTPUT_DIRECTORY.iterdir()))

Reset directory: /content/ServerMachineDataset
Children after reset: []


In [18]:
# Find the LICENSE metadata from the top-level payload

license_item = None

for item in top_level_items:
    if item.get("name") == "LICENSE":
        license_item = item
        break

print(license_item)

{'name': 'LICENSE', 'path': 'ServerMachineDataset/LICENSE', 'sha': 'a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'size': 1072, 'url': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/contents/ServerMachineDataset/LICENSE?ref=master', 'html_url': 'https://github.com/NetManAIOps/OmniAnomaly/blob/master/ServerMachineDataset/LICENSE', 'git_url': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/git/blobs/a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'download_url': 'https://raw.githubusercontent.com/NetManAIOps/OmniAnomaly/master/ServerMachineDataset/LICENSE', 'type': 'file', '_links': {'self': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/contents/ServerMachineDataset/LICENSE?ref=master', 'git': 'https://api.github.com/repos/NetManAIOps/OmniAnomaly/git/blobs/a35f9e98caf3d5d6d6164ef9a303ec4216780c01', 'html': 'https://github.com/NetManAIOps/OmniAnomaly/blob/master/ServerMachineDataset/LICENSE'}}


In [19]:
# Manual single-file download test

license_download_url = license_item["download_url"]
license_output_path = LOCAL_OUTPUT_DIRECTORY / "LICENSE"

file_response = http_session.get(license_download_url, timeout=60)
file_response.raise_for_status()

license_output_path.write_bytes(file_response.content)

print("Saved:", license_output_path.resolve())
print("Exists:", license_output_path.exists())
print("Size:", license_output_path.stat().st_size)

Saved: /content/ServerMachineDataset/LICENSE
Exists: True
Size: 1072


In [20]:
# Safer helper: always print what is being traversed and written

def recursively_download_github_directory_verbose(
    directory_path_in_repository: str,
    local_output_directory: Path,
) -> None:
    github_items = request_github_json(directory_path_in_repository)

    print(f"\nTraversing repository path: {directory_path_in_repository}")
    print(f"Local output directory: {local_output_directory}")

    if not isinstance(github_items, list):
        raise TypeError(
            f"Expected a list for directory path {directory_path_in_repository}, "
            f"but received {type(github_items).__name__}: {github_items}"
        )

    local_output_directory.mkdir(parents=True, exist_ok=True)

    for github_item in github_items:
        github_item_type = github_item["type"]
        github_item_name = github_item["name"]
        github_item_path = github_item["path"]

        print(
            "Handling:",
            "| type =", github_item_type,
            "| name =", github_item_name,
            "| path =", github_item_path,
        )

        if github_item_type == "dir":
            recursively_download_github_directory_verbose(
                directory_path_in_repository=github_item_path,
                local_output_directory=local_output_directory / github_item_name,
            )

        elif github_item_type == "file":
            github_download_url = github_item["download_url"]
            local_output_file_path = local_output_directory / github_item_name

            print("Downloading file from:", github_download_url)
            print("Writing to:", local_output_file_path)

            file_response = http_session.get(github_download_url, timeout=60)
            file_response.raise_for_status()
            local_output_file_path.parent.mkdir(parents=True, exist_ok=True)
            local_output_file_path.write_bytes(file_response.content)

            print("Saved file:", local_output_file_path)

        else:
            raise ValueError(
                f"Unsupported GitHub item type: {github_item_type} "
                f"for path {github_item_path}"
            )

In [21]:
# Reset output directory again before running the verbose traversal

import shutil

if LOCAL_OUTPUT_DIRECTORY.exists():
    shutil.rmtree(LOCAL_OUTPUT_DIRECTORY)

LOCAL_OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Reset directory:", LOCAL_OUTPUT_DIRECTORY.resolve())

Reset directory: /content/ServerMachineDataset


In [22]:
# Run the verbose traversal

recursively_download_github_directory_verbose(
    directory_path_in_repository=DATASET_DIRECTORY_IN_REPOSITORY,
    local_output_directory=LOCAL_OUTPUT_DIRECTORY,
)


Traversing repository path: ServerMachineDataset
Local output directory: ServerMachineDataset
Handling: | type = file | name = LICENSE | path = ServerMachineDataset/LICENSE
Writing to: ServerMachineDataset/LICENSE
Saved file: ServerMachineDataset/LICENSE
Handling: | type = dir | name = interpretation_label | path = ServerMachineDataset/interpretation_label

Traversing repository path: ServerMachineDataset/interpretation_label
Local output directory: ServerMachineDataset/interpretation_label
Handling: | type = file | name = machine-1-1.txt | path = ServerMachineDataset/interpretation_label/machine-1-1.txt
Writing to: ServerMachineDataset/interpretation_label/machine-1-1.txt
Saved file: ServerMachineDataset/interpretation_label/machine-1-1.txt
Handling: | type = file | name = machine-1-2.txt | path = ServerMachineDataset/interpretation_label/machine-1-2.txt
Writing to: ServerMachineDataset/interpretation_label/machine-1-2.txt
Saved file: ServerMachineDataset/interpretation_label/machine

# Pre-process into time windows aka time sequences